In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
import warnings
warnings.filterwarnings("ignore")
print("Libraries loaded ✅")

Libraries loaded ✅


In [5]:
# Write your code here
train = pd.read_csv("C:/Users/sai/Downloads/train_vitals.csv")
test  = pd.read_csv("C:/Users/sai/Downloads/test_vitals.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("\nAll columns:\n", train.columns.tolist())
print("\nSample:\n")
train.head()

Train shape: (550583, 6)
Test shape : (122347, 6)

All columns:
 ['case_id', 'time_sec', 'HR', 'MBP', 'SpO2', 'Temp']

Sample:



,case_id,time_sec,HR,MBP,SpO2,Temp
0,1,1,88.0,-8.0,96.0,NaN
1,1,3,87.0,-8.0,96.0,NaN
2,1,5,88.0,-8.0,97.0,NaN
3,1,7,88.0,-9.0,96.0,NaN
4,1,9,88.0,-9.0,96.0,NaN


In [6]:
# Write your code here
# ── Detects the real column names regardless of how sciFI renamed them ──

def detect_col(df, candidates):
    """Return first column name from candidates that exists in df."""
    cols = [c.lower() for c in df.columns]
    col_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return col_map[cand.lower()]
    return None

# Patient & time identifiers
PID_COL  = detect_col(train, ["patient_id","case_id","caseid","pid","id"])
TIME_COL = detect_col(train, ["timestamp","time","t","time_sec","time_s"])

# VitalDB vital sign column names (multiple naming conventions covered)
HR_COL   = detect_col(train, ["heart_rate","hr","Solar8000/HR","HR"])
SPO2_COL = detect_col(train, ["spo2","SpO2","Solar8000/SpO2","SNUADC/SpO2"])
SBP_COL  = detect_col(train, ["sbp","systolic_bp","nibp_sbp","abp_sbp",
                               "Solar8000/NIBP_SBP","Solar8000/ABP_SBP"])
DBP_COL  = detect_col(train, ["dbp","diastolic_bp","nibp_dbp","abp_dbp",
                               "Solar8000/NIBP_DBP","Solar8000/ABP_DBP"])
MAP_COL  = detect_col(train, ["map","mbp","nibp_mbp","abp_mbp","mean_bp",
                               "Solar8000/NIBP_MBP","Solar8000/ABP_MBP"])
TEMP_COL = detect_col(train, ["temperature","temp","bt","body_temp",
                               "Primus/TEMP1","Solar8000/BT"])

# Collect whichever vitals were found
VITAL_COLS = [c for c in [HR_COL, SPO2_COL, SBP_COL, DBP_COL, MAP_COL, TEMP_COL]
              if c is not None]

print("Patient ID col :", PID_COL)
print("Timestamp col  :", TIME_COL)
print("Vital columns  :", VITAL_COLS)
print(f"\n{len(VITAL_COLS)} vital signs detected ")

Patient ID col : case_id
Timestamp col  : time_sec
Vital columns  : ['HR', 'SpO2', 'MBP', 'Temp']

4 vital signs detected 


In [7]:
# Write your code here
# ── These are ICU clinical thresholds — hard medical knowledge ──
# Scores based on how far each reading is from the safe zone

CLINICAL_RANGES = {}

if HR_COL:
    CLINICAL_RANGES[HR_COL]   = {"low": 40,  "ok_low": 60,  "ok_high": 100, "high": 150}
if SPO2_COL:
    CLINICAL_RANGES[SPO2_COL] = {"low": 85,  "ok_low": 95,  "ok_high": 100, "high": 101}
if SBP_COL:
    CLINICAL_RANGES[SBP_COL]  = {"low": 70,  "ok_low": 90,  "ok_high": 140, "high": 200}
if DBP_COL:
    CLINICAL_RANGES[DBP_COL]  = {"low": 40,  "ok_low": 60,  "ok_high": 90,  "high": 130}
if MAP_COL:
    CLINICAL_RANGES[MAP_COL]  = {"low": 50,  "ok_low": 65,  "ok_high": 110, "high": 160}
if TEMP_COL:
    CLINICAL_RANGES[TEMP_COL] = {"low": 34.0,"ok_low": 36.0,"ok_high": 38.0,"high": 40.0}

def clinical_score_single(value, rng):
    """Returns 0 (normal) to 1 (critical) based on ICU thresholds."""
    if pd.isna(value):
        return 0.3  # missing = mildly suspicious
    lo, ok_lo, ok_hi, hi = rng["low"], rng["ok_low"], rng["ok_high"], rng["high"]
    if ok_lo <= value <= ok_hi:
        return 0.0
    elif value < lo or value > hi:
        return 1.0
    elif value < ok_lo:
        return (ok_lo - value) / (ok_lo - lo + 1e-8)
    else:
        return (value - ok_hi) / (hi - ok_hi + 1e-8)

def compute_clinical_scores(df):
    scores = np.zeros(len(df))
    for col, rng in CLINICAL_RANGES.items():
        col_scores = df[col].apply(lambda v: clinical_score_single(v, rng)).values
        scores = np.maximum(scores, col_scores)   # worst vital = patient score
    return scores

train_clinical = compute_clinical_scores(train)
test_clinical  = compute_clinical_scores(test)

print(f"Clinical scores computed — test mean: {test_clinical.mean():.4f}")

Clinical scores computed — test mean: 0.3317


In [8]:
def engineer_per_patient(df, vital_cols, pid_col, time_col, window=15):
    df = df.copy()
    
    sort_cols = [c for c in [pid_col, time_col] if c is not None]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    
    new_feat_cols = []
    
    for col in vital_cols:
        # FIXED: use .ffill() instead of fillna(method="ffill")
        if pid_col:
            df[col] = df.groupby(pid_col)[col].transform(
                lambda x: x.ffill().fillna(x.median())
            )
        else:
            df[col] = df[col].ffill().fillna(df[col].median())
        
        def rolling_feats(group):
            r_mean  = group.rolling(window, min_periods=2).mean()
            r_std   = group.rolling(window, min_periods=2).std().fillna(0)
            r_min   = group.rolling(window, min_periods=2).min()
            r_max   = group.rolling(window, min_periods=2).max()
            lag1    = group.shift(1).fillna(group.mean())
            lag5    = group.shift(5).fillna(group.mean())
            diff1   = group.diff(1).fillna(0)
            diff5   = group.diff(5).fillna(0)
            p_mean  = group.mean()
            p_std   = group.std() + 1e-8
            zscore  = (group - p_mean) / p_std

            return pd.DataFrame({
                f"{col}_rmean" : r_mean,
                f"{col}_rstd"  : r_std,
                f"{col}_rmin"  : r_min,
                f"{col}_rmax"  : r_max,
                f"{col}_range" : r_max - r_min,
                f"{col}_lag1"  : lag1,
                f"{col}_lag5"  : lag5,
                f"{col}_diff1" : diff1,
                f"{col}_diff5" : diff5,
                f"{col}_pz"    : zscore,
            })
        
        if pid_col:
            feat_df = df.groupby(pid_col)[col].apply(rolling_feats).reset_index(level=0, drop=True)
        else:
            feat_df = rolling_feats(df[col])
        
        feat_df.index = df.index
        for fc in feat_df.columns:
            df[fc] = feat_df[fc]
            new_feat_cols.append(fc)
    
    all_feat_cols = vital_cols + new_feat_cols
    df[all_feat_cols] = df[all_feat_cols].fillna(0)
    return df, all_feat_cols

print("Engineering features per patient...")
train_fe, feat_cols = engineer_per_patient(train, VITAL_COLS, PID_COL, TIME_COL)
test_fe,  _         = engineer_per_patient(test,  VITAL_COLS, PID_COL, TIME_COL)

print(f"Feature count: {len(feat_cols)}")
print(f"Train: {train_fe[feat_cols].shape}, Test: {test_fe[feat_cols].shape}")

Engineering features per patient...
Feature count: 44
Train: (550583, 44), Test: (122347, 44)


In [9]:
# Write your code here
scaler  = RobustScaler()
X_train = scaler.fit_transform(train_fe[feat_cols])
X_test  = scaler.transform(test_fe[feat_cols])

# Replace any remaining NaN/Inf
X_train = np.nan_to_num(X_train, nan=0.0, posinf=3.0, neginf=-3.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=3.0, neginf=-3.0)

print("Scaling done  X_train:", X_train.shape, "| X_test:", X_test.shape)

Scaling done  X_train: (550583, 44) | X_test: (122347, 44)


In [10]:
# Write your code here
print("Training Isolation Forest...")
iso = IsolationForest(
    n_estimators=400,
    contamination=0.05,
    max_samples=min(1024, len(X_train)),
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train)

iso_test  = -iso.decision_function(X_test)   # higher = more anomalous
iso_train = -iso.decision_function(X_train)

def minmax_norm(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

iso_test_norm  = minmax_norm(iso_test)
iso_train_norm = minmax_norm(iso_train)
print(f"IsoForest  test mean={iso_test_norm.mean():.4f}")

Training Isolation Forest...
IsoForest  test mean=0.0876


In [11]:
# LOF is too slow for 500k rows — replaced with faster OCSVM + sampling trick

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest

print("Training fast anomaly models on 500k rows...")

# ── Fast Alternative 1: One-Class SVM on a sample ──
# Train on 10k sample, predict on all
sample_size = 10000
idx = np.random.choice(len(X_train), sample_size, replace=False)
X_sample = X_train[idx]

ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm.fit(X_sample)

ocsvm_test_scores  = -ocsvm.decision_function(X_test)
lof_test_norm      = minmax_norm(ocsvm_test_scores)   # same variable name so rest of code works

print(f"OCSVM ✅  test mean={lof_test_norm.mean():.4f}")

# ── Fast Alternative 2: Second IsoForest with different params ──
iso2 = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples=2048,       # key: limits samples per tree = fast
    random_state=123,
    n_jobs=-1
)
iso2.fit(X_train)
iso2_test_norm = minmax_norm(-iso2.decision_function(X_test))
print(f"IsoForest2 ✅  test mean={iso2_test_norm.mean():.4f}")

Training fast anomaly models on 500k rows...
OCSVM ✅  test mean=0.1884
IsoForest2 ✅  test mean=0.0824


In [14]:
import numpy as np

# ── Step 1: Temporal scores (was never computed) ──
print("Computing temporal scores...")

def per_patient_rolling_anomaly(df, vital_cols, pid_col, window=20):
    scores = np.zeros(len(df))
    for col in vital_cols:
        def patient_rolling_z(group):
            rm  = group.rolling(window, min_periods=3).mean().fillna(group.mean())
            rs  = group.rolling(window, min_periods=3).std().fillna(1.0) + 1e-8
            return np.abs((group - rm) / rs)
        if pid_col:
            z = df.groupby(pid_col)[col].apply(patient_rolling_z)
            z = z.reset_index(level=0, drop=True).reindex(df.index).fillna(0)
        else:
            z = patient_rolling_z(df[col]).fillna(0)
        scores = np.maximum(scores, np.clip(z.values, 0, 5) / 5.0)
    return scores

temporal_test = per_patient_rolling_anomaly(test_fe, VITAL_COLS, PID_COL)
print(f"Temporal ✅  mean={temporal_test.mean():.4f}")

# ── Step 2: Calibration function ──
def minmax_norm(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def calibrate(s, alpha=2.5):
    s = minmax_norm(s)
    return np.power(s, 1.0 / alpha)

# ── Step 3: Final Ensemble (no autoencoder needed) ──
print("Building ensemble...")

ensemble = (
    0.35 * iso_test_norm                  +
    0.25 * lof_test_norm                  +   # OCSVM
    0.15 * iso2_test_norm                 +
    0.15 * minmax_norm(test_clinical)     +
    0.10 * minmax_norm(temporal_test)
)

final_scores = calibrate(ensemble)

print(f"\nEnsemble done ✅")
print(f"  min  = {final_scores.min():.4f}")
print(f"  max  = {final_scores.max():.4f}")
print(f"  mean = {final_scores.mean():.4f}")
print(f"  std  = {final_scores.std():.4f}")

# ── Step 4: Build & Save Submission ──
print("\nBuilding submission...")

assert len(final_scores) == len(test), \
    f"❌ Score count {len(final_scores)} != test rows {len(test)}"

submission = pd.DataFrame({
    "case_id"      : test["case_id"].values,
    "time_sec"     : test["time_sec"].values,
    "anomaly_score": np.round(final_scores, 6)
})

submission.to_csv("submission.csv", index=False)

# ── Step 5: Verify ──
verify = pd.read_csv("submission.csv")
print("\n" + "=" * 40)
print("   SUBMISSION VERIFICATION")
print("=" * 40)
print(f"  Rows          : {len(verify)}")
print(f"  Columns       : {verify.columns.tolist()}")
print(f"  Score range   : [{verify['anomaly_score'].min():.4f}, {verify['anomaly_score'].max():.4f}]")
print(f"  Null values   : {verify.isnull().sum().sum()}")
print(f"  All in [0,1]  : {verify['anomaly_score'].between(0,1).all()}")
print("=" * 40)
print(verify.head(5))
print("\n✅ SUBMISSION READY — Download submission.csv and upload!" 
      if verify['anomaly_score'].between(0,1).all() else "❌ Fix needed!")

Computing temporal scores...
Temporal ✅  mean=0.2904
Building ensemble...

Ensemble done ✅
  min  = 0.0000
  max  = 1.0000
  mean = 0.4418
  std  = 0.1203

Building submission...

   SUBMISSION VERIFICATION
  Rows          : 122347
  Columns       : ['case_id', 'time_sec', 'anomaly_score']
  Score range   : [0.0000, 1.0000]
  Null values   : 0
  All in [0,1]  : True
   case_id  time_sec  anomaly_score
0      135         1       0.981558
1      135         3       0.765130
2      135         5       0.711025
3      135         7       0.713172
4      135         9       0.708296

✅ SUBMISSION READY — Download submission.csv and upload!


In [12]:
W = {
    "iso"      : 0.30,
    "lof"      : 0.20,   # now = OCSVM
    "iso2"     : 0.10,   # new
    "ae"       : 0.20,
    "clinical" : 0.12,
    "temporal" : 0.08,
}

ensemble = (
    W["iso"]      * iso_test_norm        +
    W["lof"]      * lof_test_norm        +   # OCSVM scores
    W["iso2"]     * iso2_test_norm       +
    W["ae"]       * ae_test_norm         +
    W["clinical"] * minmax_norm(test_clinical) +
    W["temporal"] * minmax_norm(temporal_test)
)

final_scores = calibrate(ensemble)
print(f"Ensemble done ✅  mean={final_scores.mean():.4f}  std={final_scores.std():.4f}")

NameError: name 'ae_test_norm' is not defined

In [ ]:
# Write your code here
print("Training LOF...")
lof = LocalOutlierFactor(
    n_neighbors=30,
    contamination=0.05,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_train)

lof_test_norm = minmax_norm(-lof.decision_function(X_test))
print(f"LOF  test mean={lof_test_norm.mean():.4f}")

Training LOF...


In [1]:
# Write your code here
print("Training Autoencoder (MLP)...")

# Simple dense autoencoder via sklearn MLPRegressor
# Input → bottleneck → reconstruct input
# High reconstruction error = anomaly

n_feats = X_train.shape[1]
hidden  = max(8, n_feats // 4)   # bottleneck size

ae = MLPRegressor(
    hidden_layer_sizes=(hidden * 2, hidden, hidden * 2),
    activation="relu",
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    tol=1e-4
)
ae.fit(X_train, X_train)   # reconstruct input = autoencoder

# Reconstruction error per sample
ae_recon_test  = np.mean((X_test  - ae.predict(X_test))**2,  axis=1)
ae_recon_train = np.mean((X_train - ae.predict(X_train))**2, axis=1)

# Normalize using train error distribution (important!)
ae_mu, ae_sigma = ae_recon_train.mean(), ae_recon_train.std() + 1e-8
ae_test_norm = np.clip((ae_recon_test - ae_mu) / ae_sigma, 0, 5) / 5.0

print(f"Autoencoder   test mean={ae_test_norm.mean():.4f}")

Training Autoencoder (MLP)...


NameError: name 'X_train' is not defined

In [15]:
# Write your code here
# ── Weights chosen based on strength of each signal ──
# IsoForest & Autoencoder handle global patterns
# Clinical rules handle domain-specific extremes
# LOF handles local density
# Temporal handles sudden patient-level shifts

W = {
    "iso"      : 0.30,
    "lof"      : 0.20,
    "ae"       : 0.25,
    "clinical" : 0.15,
    "temporal" : 0.10,
}

ensemble = (
    W["iso"]      * iso_test_norm        +
    W["lof"]      * lof_test_norm        +
    W["ae"]       * ae_test_norm         +
    W["clinical"] * minmax_norm(test_clinical) +
    W["temporal"] * minmax_norm(temporal_test)
)

# Final smooth calibration — push scores toward useful range
# Sigmoid-like stretch to avoid everything clustering at 0.3-0.5
def calibrate(s, alpha=2.5):
    s = minmax_norm(s)
    # Mild power transform to separate scores more clearly
    return np.power(s, 1.0 / alpha)

final_scores = calibrate(ensemble)

print("Ensemble done")
print(f"  Min  : {final_scores.min():.4f}")
print(f"  Max  : {final_scores.max():.4f}")
print(f"  Mean : {final_scores.mean():.4f}")
print(f"  Std  : {final_scores.std():.4f}")

NameError: name 'ae_test_norm' is not defined

In [ ]:
# Write your code here
# Find the row ID column in test
id_col = detect_col(test, ["id","row_id","index","case_id","patient_id","timestamp"])

if id_col:
    submission = pd.DataFrame({
        id_col         : test[id_col].values,
        "anomaly_score": np.round(final_scores, 6)
    })
else:
    submission = pd.DataFrame({
        "id"           : np.arange(len(final_scores)),
        "anomaly_score": np.round(final_scores, 6)
    })

submission.to_csv("submission.csv", index=False)

print("submission.csv saved ")
print(f"Rows: {len(submission)}")
print(submission.head(10))

In [16]:
# Write your code here
submission = pd.DataFrame({
    "case_id"      : test["case_id"].values,
    "time_sec"     : test["time_sec"].values,
    "anomaly_score": np.round(final_scores, 6)
})

submission.to_csv("submission.csv", index=False)

print("submission.csv saved ✅")
print(f"Rows: {len(submission)}")
print(submission.head(10))

submission.csv saved ✅
Rows: 122347
   case_id  time_sec  anomaly_score
0      135         1       0.981558
1      135         3       0.765130
2      135         5       0.711025
3      135         7       0.713172
4      135         9       0.708296
5      135        11       0.746619
6      135        13       0.731493
7      135        15       0.727890
8      135        17       0.724092
9      135        19       0.721807


In [ ]:
# Write your code here
from IPython.display import FileLink
FileLink("submission.csv")

In [18]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import RobustScaler

print("=" * 50)
print("  IMPROVED ANOMALY DETECTION — VitalDB")
print("=" * 50)

# ── EXACT column names confirmed from your data ──
PID_COL   = "case_id"
TIME_COL  = "time_sec"
HR_COL    = "HR"
MBP_COL   = "MBP"
SPO2_COL  = "SpO2"
TEMP_COL  = "Temp"
VITAL_COLS = [HR_COL, MBP_COL, SPO2_COL, TEMP_COL]

# ════════════════════════════════════════════
# BLOCK 1: STRONG CLINICAL SCORING
# (exact ICU thresholds for these 4 vitals)
# ════════════════════════════════════════════
print("\n[1/6] Computing clinical anomaly scores...")

def clinical_score(df):
    scores = np.zeros(len(df))
    
    # HR: normal 60-100, danger <40 or >130
    if HR_COL in df.columns:
        hr = df[HR_COL].fillna(80)
        hr_score = np.where(hr < 40,  1.0,
                   np.where(hr > 130, 1.0,
                   np.where(hr < 50,  0.8,
                   np.where(hr > 120, 0.8,
                   np.where(hr < 60,  0.4,
                   np.where(hr > 100, 0.3, 0.0))))))
        scores = np.maximum(scores, hr_score)

    # SpO2: normal >95, critical <90
    if SPO2_COL in df.columns:
        spo2 = df[SPO2_COL].fillna(98)
        spo2_score = np.where(spo2 < 85,  1.0,
                     np.where(spo2 < 90,  0.9,
                     np.where(spo2 < 94,  0.5,
                     np.where(spo2 < 95,  0.2, 0.0))))
        scores = np.maximum(scores, spo2_score)

    # MBP: normal 65-110, critical <50 or >140
    if MBP_COL in df.columns:
        mbp = df[MBP_COL].fillna(85)
        mbp_score = np.where(mbp < 50,  1.0,
                    np.where(mbp > 140, 1.0,
                    np.where(mbp < 60,  0.7,
                    np.where(mbp > 120, 0.6,
                    np.where(mbp < 65,  0.3,
                    np.where(mbp > 110, 0.2, 0.0))))))
        scores = np.maximum(scores, mbp_score)

    # Temp: normal 36-38, critical <34 or >39.5
    if TEMP_COL in df.columns:
        temp = df[TEMP_COL].fillna(37)
        # Many Temp values are NaN in VitalDB — weight less
        temp_score = np.where(temp < 34.0, 1.0,
                     np.where(temp > 39.5, 1.0,
                     np.where(temp < 35.0, 0.7,
                     np.where(temp > 38.5, 0.5,
                     np.where(temp < 36.0, 0.2, 0.0)))))
        temp_score = np.where(df[TEMP_COL].isna(), 0.0, temp_score)
        scores = np.maximum(scores, temp_score)

    # ── COMBINED VITAL CRISIS: SpO2 low + HR high simultaneously ──
    if HR_COL in df.columns and SPO2_COL in df.columns:
        crisis = ((df[SPO2_COL].fillna(98) < 92) &
                  (df[HR_COL].fillna(80)   > 100)).astype(float)
        scores = np.maximum(scores, crisis * 0.95)

    # ── Hypotension crisis: very low MBP ──
    if MBP_COL in df.columns and HR_COL in df.columns:
        shock = ((df[MBP_COL].fillna(85) < 55) &
                 (df[HR_COL].fillna(80)  > 90)).astype(float)
        scores = np.maximum(scores, shock * 1.0)

    return scores

test_clinical  = clinical_score(test)
train_clinical = clinical_score(train)
print(f"   Clinical done — test mean: {test_clinical.mean():.4f}")

# ════════════════════════════════════════════
# BLOCK 2: PER-PATIENT DEVIATION SCORE
# (how unusual is this reading for THIS patient)
# ════════════════════════════════════════════
print("\n[2/6] Computing per-patient deviation scores...")

def per_patient_deviation(df, vital_cols, pid_col):
    all_scores = []
    
    for pid, group in df.groupby(pid_col):
        g_scores = np.zeros(len(group))
        
        for col in vital_cols:
            vals = group[col].fillna(group[col].median())
            if vals.std() < 1e-6:
                continue
            
            p_mean = vals.mean()
            p_std  = vals.std() + 1e-8
            
            # Per-patient z-score
            z = np.abs((vals - p_mean) / p_std)
            
            # Rolling deviation (sudden changes)
            roll_mean = vals.rolling(10, min_periods=2).mean().fillna(p_mean)
            roll_std  = vals.rolling(10, min_periods=2).std().fillna(p_std) + 1e-8
            roll_z    = np.abs((vals - roll_mean) / roll_std)
            
            col_score = np.maximum(
                np.clip(z.values, 0, 5) / 5.0,
                np.clip(roll_z.values, 0, 5) / 5.0
            )
            g_scores = np.maximum(g_scores, col_score)
        
        all_scores.append(pd.Series(g_scores, index=group.index))
    
    result = pd.concat(all_scores).reindex(df.index).fillna(0)
    return result.values

test_deviation  = per_patient_deviation(test,  VITAL_COLS, PID_COL)
print(f"   Deviation done — test mean: {test_deviation.mean():.4f}")

# ════════════════════════════════════════════
# BLOCK 3: ML MODELS
# ════════════════════════════════════════════
print("\n[3/6] Engineering ML features...")

def make_features(df, vital_cols, pid_col):
    feats = []
    
    for col in vital_cols:
        filled = df.groupby(pid_col)[col].transform(
            lambda x: x.ffill().fillna(x.median()).fillna(0)
        )
        
        p_mean = df.groupby(pid_col)[col].transform("mean").fillna(0)
        p_std  = df.groupby(pid_col)[col].transform("std").fillna(1) + 1e-8
        
        feats.append(filled.values)
        feats.append(((filled - p_mean) / p_std).values)  # per-patient z
        feats.append(filled.diff(1).fillna(0).values)     # rate of change
        feats.append(filled.diff(5).fillna(0).values)
    
    return np.column_stack(feats)

X_tr = make_features(train, VITAL_COLS, PID_COL)
X_te = make_features(test,  VITAL_COLS, PID_COL)

X_tr = np.nan_to_num(X_tr, nan=0, posinf=5, neginf=-5)
X_te = np.nan_to_num(X_te, nan=0, posinf=5, neginf=-5)

scaler = RobustScaler()
X_tr   = scaler.fit_transform(X_tr)
X_te   = scaler.transform(X_te)
print(f"   Features: {X_tr.shape[1]} cols")

print("\n[4/6] Training IsoForest x2...")
iso1 = IsolationForest(n_estimators=500, contamination=0.05,
                       max_samples=2048, random_state=42, n_jobs=-1)
iso1.fit(X_tr)
iso1_scores = minmax_norm(-iso1.decision_function(X_te))

iso2 = IsolationForest(n_estimators=300, contamination=0.08,
                       max_samples=4096, random_state=99, n_jobs=-1)
iso2.fit(X_tr)
iso2_scores = minmax_norm(-iso2.decision_function(X_te))
print(f"   IsoForest1 mean={iso1_scores.mean():.4f} | IsoForest2 mean={iso2_scores.mean():.4f}")

print("\n[5/6] Training OCSVM (sampled)...")
idx    = np.random.choice(len(X_tr), 15000, replace=False)
ocsvm  = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm.fit(X_tr[idx])
ocsvm_scores = minmax_norm(-ocsvm.decision_function(X_te))
print(f"   OCSVM mean={ocsvm_scores.mean():.4f}")

# ════════════════════════════════════════════
# BLOCK 4: ENSEMBLE + SMART CALIBRATION
# ════════════════════════════════════════════
print("\n[6/6] Building ensemble...")

def minmax_norm(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

# Clinical gets HIGH weight — it's ground truth medical knowledge
ensemble = (
    0.30 * minmax_norm(test_clinical)   +   # medical rules — most reliable
    0.25 * minmax_norm(test_deviation)  +   # per-patient temporal
    0.20 * iso1_scores                  +
    0.15 * iso2_scores                  +
    0.10 * ocsvm_scores
)

# ── Smart calibration: stretch the distribution ──
# This separates anomalies from normals much more clearly
raw = minmax_norm(ensemble)

# Percentile-based stretch for better AUC
p5  = np.percentile(raw, 5)
p95 = np.percentile(raw, 95)
stretched = np.clip((raw - p5) / (p95 - p5 + 1e-8), 0, 1)

final_scores = stretched
print(f"\n   Final score distribution:")
print(f"   min={final_scores.min():.4f} | max={final_scores.max():.4f}")
print(f"   mean={final_scores.mean():.4f} | std={final_scores.std():.4f}")

# ── Build submission ──
submission = pd.DataFrame({
    "case_id"      : test["case_id"].values,
    "time_sec"     : test["time_sec"].values,
    "anomaly_score": np.round(final_scores, 6)
})

submission.to_csv("submissions.csv", index=False)

verify = pd.read_csv("submissions.csv")
print("\n" + "=" * 40)
print(f"  Rows        : {len(verify)}")
print(f"  Columns     : {verify.columns.tolist()}")
print(f"  Range       : [{verify['anomaly_score'].min():.4f}, {verify['anomaly_score'].max():.4f}]")
print(f"  Nulls       : {verify.isnull().sum().sum()}")
print("=" * 40)
print(verify.head())
print("\n✅ READY TO SUBMIT!")

  IMPROVED ANOMALY DETECTION — VitalDB

[1/6] Computing clinical anomaly scores...
   Clinical done — test mean: 0.2759

[2/6] Computing per-patient deviation scores...
   Deviation done — test mean: 0.3392

[3/6] Engineering ML features...
   Features: 16 cols

[4/6] Training IsoForest x2...
   IsoForest1 mean=0.0452 | IsoForest2 mean=0.0407

[5/6] Training OCSVM (sampled)...
   OCSVM mean=0.1358

[6/6] Building ensemble...

   Final score distribution:
   min=0.0000 | max=1.0000
   mean=0.3047 | std=0.2842

  Rows        : 122347
  Columns     : ['case_id', 'time_sec', 'anomaly_score']
  Range       : [0.0000, 1.0000]
  Nulls       : 0
   case_id  time_sec  anomaly_score
0      135         1            1.0
1      135         3            1.0
2      135         5            1.0
3      135         7            1.0
4      135         9            1.0

✅ READY TO SUBMIT!
